
from https://www.jeremykun.com/2012/01/15/word-segmentation/


Taken from Kun (2012) and the memoization code was mostly generated with the help of AI.

References 

Kun, J. (2012, January 15). Word Segmentation, or Makingsenseofthis. Math ∩ Programming. https://www.jeremykun.com/2012/01/15/word-segmentation/


In [1]:
def splitPairs(word):
  return [(word[:i+1], word[i+1:]) for i in range(len(word))]

splitPairs("hello")

[('h', 'ello'), ('he', 'llo'), ('hel', 'lo'), ('hell', 'o'), ('hello', '')]

In [3]:
from pathlib import Path
from urllib.request import urlretrieve

NGRAM_URL = "https://norvig.com/ngrams/count_1w.txt"
NGRAM_FILE = Path("one-grams.txt")

if not NGRAM_FILE.exists():
    urlretrieve(NGRAM_URL, NGRAM_FILE)



In [13]:
import utils.monitoring as monitoring

monitoring.check_file_size(NGRAM_FILE, expected_size_mb=100)

File size is 4.73 MB, which is smaller than the expected size of 100 MB.


In [ ]:
import functools
import math

class OneGramDist(dict):
  def __init__(self):
    self.gramCount = 0
    for line in open('one-grams.txt'):
        (word, count) = line[:-1].split('\t')
        self[word] = int(count)
        self.gramCount += self[word]

  def __call__(self, word):
    if word in self:
        return float(self[word]) / self.gramCount
    else:
        # PENALTY for unknown words; tune as needed
        # return 1.0 / self.gramCount
        return 1.0 / (self.gramCount * 10**(len(word) - 2))

singleWordProb = OneGramDist()
def wordSeqFitness(words):
  return functools.reduce(lambda x,y: x+y, (math.log10(singleWordProb(w)) for w in words))

def segment(word):
  if not word: return []
  allSegmentations = [[first] + segment(rest) for (first, rest) in splitPairs(word)]
  return max(allSegmentations, key = wordSeqFitness)

segment("hellothere")
# very expensive >3m
# segment("thequickbrownfoxjumpsoverthelazydog")

['hello', 'there']

Below Modifications were part of of AI gen code using GPT-5.4 on a prompt "thequickbrownfoxjumpsoverthelazydog seems to just return itself now". The explanation is as follows:
- there was no actual "memoization" or cache happing meaning a segmented string was not looked up and therefore no save on computation. In essence not applying dynamic programming which expects recursion and avoid re-computation by using a lookup feature.



In [ ]:
@functools.lru_cache(maxsize=None)
def _segment(word):
  """A recursive helper function for segmenting a word into its most likely sequence of words.

  AI gen suggestion.
  
  - caches results by input string.
  
  check via 
  ```python
  _segment.cache_info()
  ```
  """
  if not word:
    return ()
  all_segmentations = (
    (first,) + _segment(rest)
    for first, rest in splitPairs(word)
  )
  return max(all_segmentations, key=wordSeqFitness)

def segment_v2(word):
  return list(_segment(word))

segment_v2("hellothere")
# fails as it returns the same string cached
# resolved by updating the penalty inside the OneGramDist class to be more severe for longer words
# segment_v2("thequickbrownfoxjumpsoverthelazydog")

['hello', 'there']

In [29]:
_segment.cache_clear()

In [31]:
# segment("thequickbrownfoxjumpsoverthelazydog")
segment_v2("thequickbrownfoxjumpsoverthelazydog")
# _segment.cache_info()

['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']

In [32]:
_segment.cache_info()

CacheInfo(hits=596, misses=36, maxsize=None, currsize=36)

In [33]:
segment_v2("thelongestlistofthelongeststuffatthelongestdomainnameatlonglast.com,")

['the',
 'longest',
 'list',
 'of',
 'the',
 'longest',
 'stuff',
 'at',
 'the',
 'longest',
 'domainname',
 'at',
 'long',
 'last',
 '.com,']

In [34]:
_segment.cache_info()

CacheInfo(hits=2875, misses=104, maxsize=None, currsize=104)